# 📊 Chapter 10: In-Database Spatial Statistics & Getis-Ord $G_i^*$ Hotspot Analytics
## Formulating Spatial Econometric Equations in Pure PostgreSQL SQL

### 🎯 The Mission: Why We Are Doing This
Spatial planners in Accra need to verify whether economic vitality is clustering into significant commercial corridors. We implement the **Getis-Ord Local $G_i^*$ Hotspot Statistic** directly in SQL using spatial weights matrices ($\mathbf{W}$) without external Python/R libraries:
$$G_i^* = \frac{\sum_j w_{ij} x_j - \bar{X} \sum_j w_{ij}}{S \sqrt{\frac{n \sum_j w_{ij}^2 - (\sum_j w_{ij})^2}{n - 1}}}$$

---

### 🔑 Spatial DBA Keywords Decoded

| Term | Plain-English Explanation | Production Note |
| :--- | :--- | :--- |
| **Spatial Weights Matrix ($\mathbf{W}$)** | Matrix defining neighbor relationships ($w_{ij} = 1$ if within $1,000	ext{m}$). | Foundation for spatial econometrics in SQL. |
| **$G_i^*$ Z-Score** | Standard normal statistic ($Z \ge +2.58 \implies 99\%$ Hotspot). | Industry gold-standard for cluster verification. |

In [ ]:
from config_db import execute_sql, query_df, query_gdf, preview_map

sql_gi = """
DROP TABLE IF EXISTS accra_commercial_hotspots_gi CASCADE;
CREATE TABLE accra_commercial_hotspots_gi AS
WITH grid_data AS (
    SELECT 
        hex_id, building_coverage_ratio_pct::DOUBLE PRECISION AS x_i,
        ST_Centroid(geom) AS geom_pt, geom
    FROM accra_hex_density_profile
),
global_parameters AS (
    SELECT COUNT(*)::DOUBLE PRECISION AS n, AVG(x_i) AS x_bar, STDDEV(x_i) AS s_val FROM grid_data
),
spatial_weights_w AS (
    SELECT i.hex_id AS i_id, j.hex_id AS j_id, 1.0::DOUBLE PRECISION AS w_ij, j.x_i AS x_j
    FROM grid_data i JOIN grid_data j ON ST_DWithin(i.geom_pt, j.geom_pt, 1000.0)
),
local_aggregations AS (
    SELECT sw.i_id, SUM(sw.w_ij) AS sum_w, SUM(sw.w_ij * sw.x_j) AS sum_wx, SUM(POWER(sw.w_ij, 2)) AS sum_w_sq
    FROM spatial_weights_w sw GROUP BY sw.i_id
)
SELECT 
    g.hex_id, g.geom, g.x_i AS building_coverage_pct,
    ROUND(((la.sum_wx - (gp.x_bar * la.sum_w)) / NULLIF((gp.s_val * SQRT( ( (gp.n * la.sum_w_sq) - POWER(la.sum_w, 2) ) / (gp.n - 1.0) )), 0))::NUMERIC, 3) AS gi_z_score,
    CASE 
        WHEN (la.sum_wx - (gp.x_bar * la.sum_w)) / NULLIF((gp.s_val * SQRT( ( (gp.n * la.sum_w_sq) - POWER(la.sum_w, 2) ) / (gp.n - 1.0) )), 0) >= 2.58 THEN 'Hotspot (99% Confidence)'
        WHEN (la.sum_wx - (gp.x_bar * la.sum_w)) / NULLIF((gp.s_val * SQRT( ( (gp.n * la.sum_w_sq) - POWER(la.sum_w, 2) ) / (gp.n - 1.0) )), 0) >= 1.96 THEN 'Hotspot (95% Confidence)'
        WHEN (la.sum_wx - (gp.x_bar * la.sum_w)) / NULLIF((gp.s_val * SQRT( ( (gp.n * la.sum_w_sq) - POWER(la.sum_w, 2) ) / (gp.n - 1.0) )), 0) <= -1.96 THEN 'Coldspot (95% Confidence)'
        ELSE 'Not Statistically Significant'
    END AS hotspot_category
FROM grid_data g
JOIN local_aggregations la ON g.hex_id = la.i_id
CROSS JOIN global_parameters gp;

CREATE INDEX idx_gi_geom ON accra_commercial_hotspots_gi USING GIST(geom);
"""

execute_sql(sql_gi)
print("✅ Getis-Ord Gi* in-database spatial statistics completed.")

In [ ]:
# Summary Distribution of Statistically Significant Hotspots
df_gi = query_df("""
SELECT hotspot_category, COUNT(*) AS cell_count, ROUND(AVG(gi_z_score)::NUMERIC, 2) AS mean_z_score
FROM accra_commercial_hotspots_gi
GROUP BY hotspot_category
ORDER BY cell_count DESC;
""")

df_gi

In [ ]:
# Interactive Map Preview: Statistical Hotspots & Clusters
gdf_gi = query_gdf("""
SELECT hex_id, hotspot_category, gi_z_score, geom FROM accra_commercial_hotspots_gi;
""")

preview_map(gdf_gi, tooltip_cols=["hotspot_category", "gi_z_score"])